In [1]:
library("SeuratDisk")
library("zellkonverter")
library("Seurat")
library(tidyverse)
library(kasumi)

Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat

Registered S3 method overwritten by 'zellkonverter':
  method                                             from      
  py_to_r.pandas.core.arrays.categorical.Categorical reticulate

Lade n"otiges Paket: SeuratObject

Lade n"otiges Paket: sp

'SeuratObject' was built under R 4.4.0 but the current version is
4.4.2; it is recomended that you reinstall 'SeuratObject' as the ABI
for R may have changed


Attache Paket: 'SeuratObject'


Die folgenden Objekte sind maskiert von 'package:base':

    intersect, t


-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.5.2
v ggplot2   4.0.0     v tibble    3.3.0
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.1.0     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()


In [2]:
adata_file <- "/Users/jawadalaaedeen/Desktop/PhD/NMC/results/adata_unified.h5ad"

In [3]:
ad <- readH5AD(adata_file)


Warning message:
"The names of these selected obs columns have been modified to match R
conventions: 'Cell 3D Volume' -> 'Cell.3D.Volume', 'Cell Bbox Bottom' ->
'Cell.Bbox.Bottom', 'Cell Bbox Left' -> 'Cell.Bbox.Left', 'Cell Bbox Right' ->
'Cell.Bbox.Right', 'Cell Bbox Top' -> 'Cell.Bbox.Top', 'Cell Bbox X Size' ->
'Cell.Bbox.X.Size', 'Cell Bbox Y Size' -> 'Cell.Bbox.Y.Size', 'Cell Center X'
-> 'Cell.Center.X', 'Cell Center Y' -> 'Cell.Center.Y', 'Cell Center Y
Inverted' -> 'Cell.Center.Y.Inverted', 'Cell Contour Bending Energy' ->
'Cell.Contour.Bending.Energy', 'Cell Contour Length' -> 'Cell.Contour.Length',
'Cell Convex Hull Area' -> 'Cell.Convex.Hull.Area', 'Cell Convex Hull
Perimeter' -> 'Cell.Convex.Hull.Perimeter', 'Cell Convexity' ->
'Cell.Convexity', 'Cell Cytoplasm Area' -> 'Cell.Cytoplasm.Area', 'Cell Feret
Diameter Max' -> 'Cell.Feret.Diameter.Max', 'Cell Feret Diameter Maximum Angle'
-> 'Cell.Feret.Diameter.Maximum.Angle', ..., 'site of biopsy/resection for
MACSima sample' 

In [4]:
ad_seurat <- as.Seurat(ad, counts = "X", data = NULL)


Warning message:
"`PackageCheck()` was deprecated in SeuratObject 5.0.0.
i Please use `rlang::check_installed()` instead.
i The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>."


In [5]:
patient_exps <- ad_seurat@meta.data %>%
pull(patient_exp) %>%
unique()

cts <- ad_seurat@meta.data %>%
pull(cell_type) %>%
unique()

all.cells.ctcl <- patient_exps %>% map(\(id){
ad_seurat@meta.data %>%
filter(patient_exp == id) %>%
pull(cell_type) %>%
map(~ .x == cts) %>%
rlist::list.rbind() %>%
`colnames<-`(make.names(cts)) %>%
as_tibble(.name_repair = "unique")
})
names(all.cells.ctcl) <- patient_exps

all.positions.ctcl <- patient_exps %>% map(\(id){
ad_seurat@meta.data %>%
filter(patient_exp == id) %>%
select(Cell.Center.X, Cell.Center.Y) %>%
`colnames<-`(c("x", "y"))
})
names(all.positions.ctcl) <- patient_exps

In [6]:
as.character(patient_exps) %>% walk(\(id){
ct <- all.cells.ctcl[[id]]
pos <- all.positions.ctcl[[id]]
kasumi.views <- create_initial_view(ct) %>% add_paraview(pos, 20, family = "constant", prefix="p.")
suppressWarnings(
run_kasumi(kasumi.views, pos, window=400, overlap=50, id, "CTCLct400.sqm", minu=20, bypass.intra=TRUE, sqlite_timeout = 250)
)
})


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors per unit


Sliding


Generating paraview using 20 nearest neighbors 

In [6]:
options(future.globals.maxSize = 5 * 1024^3)  # 5 GB

In [7]:
getwd()

[1] "/Users/jawadalaaedeen/Desktop/PhD/NMC/NMC_spatial/R"

In [ ]:
kasumi.results <- collect_results("/Users/jawadalaaedeen/Desktop/PhD/NMC/results/CTCLct400.sqm")

In [68]:
# First representation - relationships
kasumi.representation <- extract_representation(kasumi.results)
# Second representation - clusters
kasumi.clusters <- extract_clusters(kasumi.representation, "leiden", 0.4, 0.9)
# Cluster composition
kasumi.agg <- aggregate_clusters(kasumi.clusters)
# Third representation - persistent cluster composition -
# clusters that are present in at least 5 samples
persistent.clusters <- persistent_clusters(kasumi.agg, 5)
kasumi.persistent <- kasumi.agg %>% select(id, all_of(persistent.clusters))

New names:
* `` -> `...1`
* `` -> `...2`
* `` -> `...3`
* `` -> `...4`
* `` -> `...5`
* `` -> `...6`
* `` -> `...7`
* `` -> `...8`
* `` -> `...9`
* `` -> `...10`
* `` -> `...11`
* `` -> `...12`
* `` -> `...13`
* `` -> `...14`
* `` -> `...15`
* `` -> `...16`
* `` -> `...17`
* `` -> `...18`
* `` -> `...19`
* `` -> `...20`
* `` -> `...21`
* `` -> `...22`
* `` -> `...23`
* `` -> `...24`
* `` -> `...25`
* `` -> `...26`
* `` -> `...27`
* `` -> `...28`
* `` -> `...29`
* `` -> `...30`
* `` -> `...31`
* `` -> `...32`
* `` -> `...33`
* `` -> `...34`


In [99]:
kasumi.views

Tumor_cells,NFC,M1_macrophages,MDSCs,Actin._cells,M1_M2_macrophages,CD8._T_cells,Lymphatic_endothelial_cells,CD4._T_cells,M2_macrophages,Endothelial_cells,Granulocytes,Treg_T_cells,DCs,B_cells,Plasma_cells,Mast_cells,NK_cells
<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE


In [98]:
kasumi.representation

id,xcenter,ycenter,paraview.10_p.Tumor_cells_Tumor_cells,paraview.10_p.NFC_Tumor_cells,paraview.10_p.M1_macrophages_Tumor_cells,paraview.10_p.MDSCs_Tumor_cells,paraview.10_p.Actin._cells_Tumor_cells,paraview.10_p.M1_M2_macrophages_Tumor_cells,paraview.10_p.CD8._T_cells_Tumor_cells,...,paraview.10_p.Plasma_cells_Endothelial_cells,paraview.10_p.NK_cells_Endothelial_cells,paraview.10_p.Tumor_cells_Plasma_cells,paraview.10_p.NFC_Plasma_cells,paraview.10_p.MDSCs_Plasma_cells,paraview.10_p.Actin._cells_Plasma_cells,paraview.10_p.CD8._T_cells_Plasma_cells,paraview.10_p.Endothelial_cells_Plasma_cells,paraview.10_p.Granulocytes_Plasma_cells,paraview.10_p.Plasma_cells_Plasma_cells
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NMC_16_ROI19,227.1,205.79,1.323841e+00,0.75734020,1.045971e+00,0.000000e+00,1.792563e+00,0,0,...,0,0,0,0,0,0,0,0,0,0
NMC_16_ROI19,227.1,1205.79,4.254637e-01,0.06254626,0.000000e+00,5.481633e-02,1.937562e-01,0,0,...,0,0,0,0,0,0,0,0,0,0
NMC_16_ROI19,227.1,1405.79,0.000000e+00,0.00000000,0.000000e+00,0.000000e+00,0.000000e+00,0,0,...,0,0,0,0,0,0,0,0,0,0
NMC_16_ROI19,227.1,3805.79,1.234447e+00,0.00000000,0.000000e+00,1.897276e+00,4.857993e-01,0,0,...,0,0,0,0,0,0,0,0,0,0
NMC_16_ROI19,227.1,4005.79,6.588233e-01,0.00000000,0.000000e+00,1.142437e+00,1.502635e-01,0,0,...,0,0,0,0,0,0,0,0,0,0
NMC_16_ROI19,427.1,205.79,7.669969e-01,0.24891104,1.626533e-01,0.000000e+00,5.731006e-01,0,0,...,0,0,0,0,0,0,0,0,0,0
NMC_16_ROI19,427.1,1005.79,2.057998e+00,0.14306444,0.000000e+00,2.313998e+00,1.212083e+00,0,0,...,0,0,0,0,0,0,0,0,0,0
NMC_16_ROI19,427.1,2805.79,2.767869e+00,0.68461448,1.012986e+00,1.871562e+00,6.331097e-01,0,0,...,0,0,0,0,0,0,0,0,0,0
NMC_16_ROI19,427.1,3005.79,2.453778e+00,0.82547896,7.985788e-01,5.237714e-01,1.141373e+00,0,0,...,0,0,0,0,0,0,0,0,0,0
